In [ ]:
import pandas as pd


df = pd.read_csv("./data/open-meteo-21.05N105.81E18m - open-meteo-21.05N105.81E18m.csv", skiprows=3)

df.head()


In [ ]:
df["time"] = pd.to_datetime(df["time"])

df["hour"] = df["time"].dt.hour
df["dayofyear"] = df["time"].dt.dayofyear
df["month"] = df["time"].dt.month

df["temp_lag_1h"] = df["temperature_2m (°C)"].shift(1)
df["temp_lag_24h"] = df["temperature_2m (°C)"].shift(24)
df["target_temp"] = df["temperature_2m (°C)"].shift(-24)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df = df.dropna()

In [ ]:
features = [
    "relative_humidity_2m (%)",
    "cloud_cover (%)",
    "wind_speed_10m (km/h)",
    "pressure_msl (hPa)",
    "precipitation (mm)",
    "hour",
    "dayofyear",
    "month",
    "temp_lag_1h",
    "temp_lag_24h"
]

# Training
X = df[features]
y = df["target_temp"]

horizon = 24
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index - horizon]
y_train = y.iloc[:split_index - horizon]

X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

In [ ]:
X.head()

In [ ]:
# Looking for promise model before fine-tune

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd
import time

num_pipeline = make_pipeline(
    StandardScaler()
)

preprocessing = ColumnTransformer([
    ("num", num_pipeline, features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "KNN": KNeighborsRegressor()
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessing", preprocessing),
        ("model", model)
    ])

    start_time = time.time()
    pipe.fit(X_train, y_train)
    training_time = time.time() - start_time
    y_pred = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Training Time (s)": training_time
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
tscv = TimeSeriesSplit(n_splits=3, gap=24)


full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("gradient_boosting", GradientBoostingRegressor(random_state=42)),
])

param_distribs = {
    "gradient_boosting__n_estimators": randint(80, 250),
    "gradient_boosting__learning_rate": uniform(0.03, 0.12),
    "gradient_boosting__max_depth": randint(2, 5),
    "gradient_boosting__min_samples_leaf": randint(1, 15),
    "gradient_boosting__subsample": uniform(0.7, 0.3),
    "gradient_boosting__max_features": [None, "sqrt", 0.7, 1.0],
}

rnd_search = RandomizedSearchCV(
    full_pipeline, 
    param_distributions=param_distribs, 
    n_iter=10, 
    cv=tscv,
    scoring='neg_root_mean_squared_error', 
    random_state=42)

rnd_search.fit(X_train, y_train)

In [ ]:
baseline_model = Pipeline([
    ("preprocessing", preprocessing),
    ("gradient_boosting", GradientBoostingRegressor(random_state=42)),
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, baseline_pred)))
print("Baseline MAE:", mean_absolute_error(y_test, baseline_pred))
print("Baseline R²:", r2_score(y_test, baseline_pred))

print("=================")

tuned_model = rnd_search.best_estimator_
tuned_pred = tuned_model.predict(X_test)

print("Tuned RMSE:", np.sqrt(mean_squared_error(y_test, tuned_pred)))
print("Tuned MAE:", mean_absolute_error(y_test, tuned_pred))
print("Tuned R²:", r2_score(y_test, tuned_pred))